<a href="https://colab.research.google.com/github/bolinocroustibat/movies-palettes/blob/main/movies_palettes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Imports

In [ ]:
import numpy as np
import cv2
import hashlib
import json
import os
import re
import requests
import time
from datetime import datetime, timezone
from dateutil import parser
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm

# Configuration constants

*   **`RESIZE_W` and `RESIZE_H` (Reduced Frame Size)**

  This determines the resolution of each frame after downsampling. A reasonable value should:
    - Retain enough detail for the color histogram.
    - Eliminate excessive computational overhead from large frame sizes.

  Suggested Value: 160 x 90 (W x H)
  - This keeps the aspect ratio of typical widescreen content (16:9).
  -	At 160x90, you get 14,400 pixels per frame, which is sufficient for a color histogram while keeping processing fast.

*  **`FRAME_SKIP` (Frames Skipped)**

    This determines how many frames you skip between processed frames.
  -	Movies at 24fps typically don’t have significant color changes frame-by-frame.
  -	A higher skip value reduces computation but might miss some rapid scene changes.

  Suggested Value: `FRAME_SKIP = 60`
 	- This processes 1 frame per 2.5 seconds (approx.), enough to capture major color changes without excessive redundancy.
 	-	For a 90-minute movie:
 	   - Total frames at 24fps:  90 x 60 x 24 = 129,600
 	   - With `FRAME_SKIP = 60`, you’ll process around  129,600 / 60 = 2,160  frames per movie.

*  **`BATCH_SIZE` (Frames Processed at Once)**

  This determines the number of frames processed in a single batch.
 	-	Larger batch sizes are computationally efficient as you can leverage batch processing in libraries like OpenCV.
 	-	However, too large a batch size might lead to memory limitations.

  Suggested Values:
  - For CPU processing, you can try `BATCH_SIZE = 20`.
  - For an NVIDIA A100 GPU, which has 40/80GB of VRAM depending on the variant, you can use a significantly larger batch, like `BATCH_SIZE = 32`



In [ ]:
PALETTE_SIZE = 8

FRAME_SKIP = 60  # Process one frame out of every FRAME_SKIP frames in order to speed up the process
# Resize the image to RESIZE_W x RESIZE_H pixels in order to reduce complexity
RESIZE_W = 160
RESIZE_H = 90
BATCH_SIZE = 32  # Number of frames to process in each batch

# Kept so the API payload still has this field. Extraction does not shift saturation.
SATURATION_FACTOR = 1.0

# Recalculate a new palette even if it already has one if the palette is older than this.
# Put None if you want to recalculate no matter what
RECALC_IF_BEFORE = None
# Calculate only a specific movie
MOVIE_TO_PROCESS = "Dick Tracy"  # Set to None to process all movies

# "api" posts palettes to the cinematheque API.
# "json" writes palettes.json in the movies folder and does not call the API.
BACKEND = "api"
API_BASE_URL = "https://cinematheque.adriencarpentier.com"
DRIVE_ROOT = Path("/content/drive/MyDrive")
MOVIES_DIR = DRIVE_ROOT / "MOVIES"
PALETTES_JSON = MOVIES_DIR / "palettes.json"

# Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# API write key

In Colab, open the key icon in the left sidebar, add a secret named `API_WRITE_KEY`, and turn on notebook access. The notebook reads it below. The key is not stored in the notebook or on Drive.

Outside Colab, the same name is read from a `.env` file or the environment. JSON mode does not need a key.

In [ ]:
def load_api_write_key() -> str | None:
    """Read API_WRITE_KEY from Colab secrets, or from .env outside Colab."""
    try:
        from google.colab import userdata
    except ImportError:
        return _read_dotenv_key()

    try:
        key = userdata.get("API_WRITE_KEY")
    except Exception:
        print("Could not read the Colab secret API_WRITE_KEY.")
        return None
    key = (key or "").strip()
    return key or None


def _read_dotenv_key() -> str | None:
    env_path = Path(".env")
    if env_path.is_file():
        for line in env_path.read_text().splitlines():
            stripped = line.strip()
            if not stripped or stripped.startswith("#") or "=" not in stripped:
                continue
            name, value = stripped.split("=", 1)
            if name.strip() == "API_WRITE_KEY":
                value = value.strip().strip('"').strip("'")
                return value or None
    value = os.environ.get("API_WRITE_KEY", "").strip()
    return value or None


API_WRITE_KEY = None
if BACKEND == "api":
    API_WRITE_KEY = load_api_write_key()
    if not API_WRITE_KEY:
        raise RuntimeError(
            'API_WRITE_KEY is missing. In Colab, add a secret named API_WRITE_KEY '
            'and allow notebook access. Or set BACKEND = "json".'
        )


# Function to get runtime type

In [ ]:
def get_runtype_type() -> str:
  if int(os.environ.get("COLAB_GPU", 0)) > 0:
    return "GPU"
  elif "TPU_DRIVER_MODE" in os.environ and os.environ["TPU_DRIVER_MODE"] == "tpu":
    return "TPU"
  elif "COLAB_TPU_ADDR" in os.environ and os.environ["COLAB_TPU_ADDR"]:
    return "TPU"
  return "unknown"

# Function to build a UUID

In [ ]:
def generate_palette_id(title: str, calculation_date: str) -> str:
    """Generate a short unique identifier for a palette"""
    unique_string = f"{title}_{calculation_date}"
    hash_object = hashlib.md5(unique_string.encode())
    return hash_object.hexdigest()[:6]

# Main logic

This is the main logic here. It might takes a few minutes to a few hours per movie file, depending on where it's run.

Reading the sampled frames is what takes the time. The palette itself is an Oklab histogram of those frames.


In [ ]:
L_BIN = 0.05
AB_BIN = 0.025
MIN_LIGHTNESS = 0.04
NEUTRAL_CHROMA = 0.03
MIN_DISTANCE = 0.07
MIN_DISTANCE_FLOOR = 0.02
DISTANCE_RELAX = 0.6
MAX_NEUTRALS = 2
NEUTRAL_MIN_SHARE = 0.05
NEUTRAL_L_GAP = 0.25
MIN_CHROMATIC_WEIGHT_SHARE = 0.005
BW_MAX_CHROMATIC_SHARE = 0.03
ACCENT_MIN_SHARE = 0.004
ACCENT_MIN_CHROMA = 0.08


def _srgb_to_linear(channel: np.ndarray) -> np.ndarray:
    return np.where(
        channel <= 0.04045,
        channel / 12.92,
        ((channel + 0.055) / 1.055) ** 2.4,
    )


def rgb_to_oklab(rgb: np.ndarray) -> np.ndarray:
    """Convert sRGB values in 0..1 to Oklab."""
    rgb = np.asarray(rgb, dtype=np.float64)
    red, green, blue = (_srgb_to_linear(rgb[..., channel]) for channel in range(3))
    long = 0.4122214708 * red + 0.5363325363 * green + 0.0514459929 * blue
    medium = 0.2119034982 * red + 0.6806995451 * green + 0.1073969566 * blue
    short = 0.0883024619 * red + 0.2817188376 * green + 0.6299787005 * blue
    long_cbrt = np.cbrt(long)
    medium_cbrt = np.cbrt(medium)
    short_cbrt = np.cbrt(short)
    lightness = (
        0.2104542553 * long_cbrt + 0.7936177850 * medium_cbrt - 0.0040720468 * short_cbrt
    )
    green_red = (
        1.9779984951 * long_cbrt - 2.4285922050 * medium_cbrt + 0.4505937099 * short_cbrt
    )
    blue_yellow = (
        0.0259040371 * long_cbrt + 0.7827717662 * medium_cbrt - 0.8086757660 * short_cbrt
    )
    return np.stack((lightness, green_red, blue_yellow), axis=-1)


def _bin_keys(oklab: np.ndarray) -> np.ndarray:
    lightness_index = np.floor(oklab[:, 0] / L_BIN).astype(np.int64)
    green_red_index = np.floor(oklab[:, 1] / AB_BIN).astype(np.int64)
    blue_yellow_index = np.floor(oklab[:, 2] / AB_BIN).astype(np.int64)
    return (
        (lightness_index + 32) * 1_000_000
        + (green_red_index + 512) * 1_000
        + (blue_yellow_index + 512)
    )


def _accumulate_histogram(histogram: dict[int, np.ndarray], frames: list[np.ndarray]) -> None:
    """Add resized BGR frames to a histogram of fine Oklab bins.

    Each bin stores a pixel count and the sum of sRGB channels, so its color is
    the mean of pixels that already look the same.
    """
    if not frames:
        return
    resized = [cv2.resize(frame, (RESIZE_W, RESIZE_H)) for frame in frames]
    rgb = np.stack([cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) for frame in resized])
    pixels = rgb.reshape(-1, 3)
    oklab = rgb_to_oklab(pixels.astype(np.float64) / 255.0)
    visible = oklab[:, 0] >= MIN_LIGHTNESS
    if not np.any(visible):
        return
    pixels = pixels[visible].astype(np.float64)
    keys = _bin_keys(oklab[visible])
    unique_keys, inverse = np.unique(keys, return_inverse=True)
    counts = np.bincount(inverse)
    sums = np.zeros((unique_keys.shape[0], 3), dtype=np.float64)
    np.add.at(sums, inverse, pixels)
    for key, count, rgb_sum in zip(unique_keys, counts, sums):
        slot = histogram.get(int(key))
        if slot is None:
            histogram[int(key)] = np.array(
                [count, rgb_sum[0], rgb_sum[1], rgb_sum[2]], dtype=np.float64
            )
        else:
            slot[0] += count
            slot[1:] += rgb_sum


In [ ]:
def _oklab_distance(left: np.ndarray, right: np.ndarray) -> float:
    delta = left - right
    return float(np.sqrt(delta @ delta))


def _pick_diverse(candidates: list[dict], slots: int) -> list[dict]:
    """Keep the heaviest colors that stay at least MIN_DISTANCE apart in Oklab.

    When a film is really one hue, the distance is relaxed so distinct
    lightnesses of that hue can still fill the palette.
    """
    picked: list[dict] = []
    remaining = list(candidates)
    distance = MIN_DISTANCE
    while len(picked) < slots and remaining and distance >= MIN_DISTANCE_FLOOR - 1e-12:
        rejected: list[dict] = []
        for candidate in remaining:
            if len(picked) >= slots:
                rejected.append(candidate)
                continue
            far_enough = all(
                _oklab_distance(candidate["oklab"], chosen["oklab"]) >= distance
                for chosen in picked
            )
            if far_enough:
                picked.append(candidate)
            else:
                rejected.append(candidate)
        if len(picked) >= slots or not rejected:
            break
        relaxed = distance * DISTANCE_RELAX
        if relaxed < MIN_DISTANCE_FLOOR:
            if distance <= MIN_DISTANCE_FLOOR + 1e-12:
                break
            distance = MIN_DISTANCE_FLOOR
        else:
            distance = relaxed
        remaining = rejected
    return picked


def _pick_neutrals(neutrals: list[dict], slots: int) -> list[dict]:
    """Up to two grays, the darkest and the lightest, in slots left by the hues."""
    qualifying = [color for color in neutrals if color["share"] >= NEUTRAL_MIN_SHARE]
    if not qualifying or slots <= 0:
        return []
    qualifying.sort(key=lambda color: color["oklab"][0])
    chosen = [qualifying[0]]
    lightest = qualifying[-1]
    lightness_gap = float(lightest["oklab"][0] - qualifying[0]["oklab"][0])
    if slots >= 2 and lightest is not qualifying[0] and lightness_gap >= NEUTRAL_L_GAP:
        chosen.append(lightest)
    return chosen


def _rgb_tuple(mean_rgb: np.ndarray) -> tuple[int, int, int]:
    clipped = np.clip(np.rint(mean_rgb), 0, 255).astype(int)
    return int(clipped[0]), int(clipped[1]), int(clipped[2])


def palette_from_histogram(
    histogram: dict[int, np.ndarray],
) -> list[tuple[int, int, int]] | None:
    """Return recognizable colors, or None when the frames are black and white.

    Chromatic bins are ranked by count times chroma squared, then kept only when
    they are far enough apart in Oklab. Neutrals fill leftover slots.
    """
    if not histogram:
        return None
    slots_data = list(histogram.values())
    counts = np.array([slot[0] for slot in slots_data], dtype=np.float64)
    total = float(counts.sum())
    if total <= 0:
        return None
    means = np.stack([slot[1:] / slot[0] for slot in slots_data])
    oklab = rgb_to_oklab(means / 255.0)
    chromas = np.hypot(oklab[:, 1], oklab[:, 2])
    shares = counts / total
    accent = (chromas >= ACCENT_MIN_CHROMA) & (shares >= ACCENT_MIN_SHARE)
    chromatic = chromas >= NEUTRAL_CHROMA
    chromatic_share = float(counts[chromatic].sum() / total)
    if not np.any(accent) and chromatic_share < BW_MAX_CHROMATIC_SHARE:
        return None

    chromatic_index = np.flatnonzero(chromatic)
    neutral_index = np.flatnonzero(~chromatic)
    chromatic_weight = counts[chromatic_index] * chromas[chromatic_index] ** 2
    weight_total = float(chromatic_weight.sum())
    candidates = []
    if weight_total > 0:
        for index, weight in zip(chromatic_index, chromatic_weight):
            if weight / weight_total < MIN_CHROMATIC_WEIGHT_SHARE:
                continue
            candidates.append(
                {
                    "rgb": means[index],
                    "oklab": oklab[index],
                    "weight": float(weight),
                    "share": float(shares[index]),
                }
            )
    candidates.sort(key=lambda color: color["weight"], reverse=True)
    picked = _pick_diverse(candidates, PALETTE_SIZE)
    neutrals = [
        {
            "rgb": means[index],
            "oklab": oklab[index],
            "share": float(shares[index]),
        }
        for index in neutral_index
    ]
    picked_neutrals = _pick_neutrals(neutrals, min(MAX_NEUTRALS, PALETTE_SIZE - len(picked)))
    picked_neutrals.sort(key=lambda color: color["oklab"][0])
    picked.sort(key=lambda color: float(np.arctan2(color["oklab"][2], color["oklab"][1])))
    return [_rgb_tuple(color["rgb"]) for color in picked_neutrals + picked]

In [ ]:
def process_batch(frames: list[np.ndarray], histogram: dict[int, np.ndarray]) -> None:
    """Add a batch of BGR frames to the running Oklab histogram."""
    _accumulate_histogram(histogram, frames)


def palette_from_frames(frames: list[np.ndarray]) -> list[tuple[int, int, int]] | None:
    """Build a palette from BGR frames, or None when they are black and white."""
    histogram: dict[int, np.ndarray] = {}
    process_batch(frames, histogram)
    return palette_from_histogram(histogram)


In [ ]:
def process_movie(
    movie_title: str, file_path: Path, progress_bar: tqdm | None = None
) -> dict | None:
    """
    Calculate the recognizable color palette of a movie.

    Sampled frames accumulate into an Oklab histogram. A black and white movie
    is skipped and this function returns None.

    Parameters:
    - file_path: Path to the movie file
    - movie_title: Movie title
    Returns:
    dict | None: The new palette entry, or None when the movie is skipped.
    """
    start_time: float = time.time()

    if progress_bar is not None:
        progress_bar.set_description(f'Processing movie "{movie_title}"...')

    cap = cv2.VideoCapture(str(file_path))
    try:
        frames_count: int = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frames_to_process: int = frames_count // FRAME_SKIP
        if frames_to_process < 1:
            print(f'Movie "{movie_title}" has less than {FRAME_SKIP} frames, skipping...')
            return None

        print(f'Movie "{movie_title}" has {frames_count} frames.')

        frame_nb = 0
        frames_processed = 0
        histogram: dict[int, np.ndarray] = {}

        with tqdm(total=frames_to_process, desc="Frames processed") as frame_pbar:
            while frame_nb < frames_count:
                batch_frames = []
                frames_read = 0

                remaining_frames = frames_to_process - frames_processed
                current_batch_size = min(BATCH_SIZE, remaining_frames)

                for _ in range(current_batch_size):
                    if frame_nb >= frames_count:
                        break

                    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_nb)
                    status, frame = cap.read()

                    if not status:
                        frame_nb += FRAME_SKIP
                        continue

                    batch_frames.append(frame)
                    frames_read += 1
                    frame_nb += FRAME_SKIP

                if not batch_frames:
                    break

                process_batch(batch_frames, histogram)
                frame_pbar.update(frames_read)
                frames_processed += frames_read
    finally:
        cap.release()

    if frames_processed == 0:
        print(f"No colors extracted for {movie_title}. Skipping.")
        return None

    colors = palette_from_histogram(histogram)
    if colors is None:
        print(f'Movie "{movie_title}" is black and white, skipping...')
        return None
    if not colors:
        print(f"No colors extracted for {movie_title}. Skipping.")
        return None

    end_time: float = time.time()
    duration: int = int(end_time - start_time)
    now: str = datetime.now(timezone.utc).isoformat()

    return {
        "id": generate_palette_id(movie_title, now),
        "calculation_date": now,
        "calculation_duration_seconds": duration,
        "is_black_and_white": False,
        "colors": colors,
        "clusters_nb": len(colors),
        "frame_skip": FRAME_SKIP,
        "resize_width": RESIZE_W,
        "resize_height": RESIZE_H,
        "batch_size": BATCH_SIZE,
        "clustering_method": "oklab-histogram",
        "color_space": "OKLAB",
        "saturation_factor": SATURATION_FACTOR,
        "runtime": get_runtype_type(),
    }


In [ ]:
# Synthetic frames, no video file. The same selector as process_movie.

def _frame_from_rgb_blocks(parts: list[tuple[float, tuple[int, int, int]]]) -> np.ndarray:
    frame = np.zeros((RESIZE_H, RESIZE_W, 3), dtype=np.uint8)
    flat = frame.reshape(-1, 3)
    pixels = flat.shape[0]
    start = 0
    for fraction, color in parts:
        count = int(round(fraction * pixels))
        flat[start : start + count] = color
        start += count
    if start < pixels:
        flat[start:] = parts[-1][1]
    return cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)


def _is_red(color: tuple[int, int, int]) -> bool:
    red, green, blue = color
    return red > 180 and red > green + 80 and red > blue + 80


def _is_blue(color: tuple[int, int, int]) -> bool:
    red, green, blue = color
    return blue > 180 and blue > red + 80 and blue > green + 40


def _is_brown(color: tuple[int, int, int]) -> bool:
    red, green, blue = color
    return red > 90 and red < 180 and green > 40 and blue < green and (red - blue) > 30


def _is_green(color: tuple[int, int, int]) -> bool:
    red, green, blue = color
    return green > red + 20 and green > blue + 10


brown_red_blue = palette_from_frames([
    _frame_from_rgb_blocks([
        (0.85, (140, 90, 50)),
        (0.075, (220, 20, 20)),
        (0.075, (20, 40, 220)),
    ])
])
assert brown_red_blue is not None
assert any(_is_red(color) for color in brown_red_blue), brown_red_blue
assert any(_is_blue(color) for color in brown_red_blue), brown_red_blue
assert sum(_is_brown(color) for color in brown_red_blue) <= 1, brown_red_blue

gray_with_red = palette_from_frames([
    _frame_from_rgb_blocks([
        (0.98, (128, 128, 128)),
        (0.02, (220, 30, 30)),
    ])
])
assert gray_with_red is not None
assert any(_is_red(color) for color in gray_with_red), gray_with_red

greens = palette_from_frames([
    _frame_from_rgb_blocks([
        (1 / 3, (10, 50, 0)),
        (1 / 3, (20, 140, 40)),
        (1 / 3, (140, 230, 150)),
    ])
])
assert greens is not None and len(greens) >= 3, greens
assert all(_is_green(color) for color in greens), greens

gradient = np.linspace(16, 240, RESIZE_W, dtype=np.uint8)
gray_frame = np.stack(
    [np.broadcast_to(gradient, (RESIZE_H, RESIZE_W))] * 3, axis=-1
).copy()
assert palette_from_frames([cv2.cvtColor(gray_frame, cv2.COLOR_RGB2BGR)]) is None

print("Palette checks passed.")
print("Brown, red and blue:", brown_red_blue)
print("Gray with a red spot:", gray_with_red)
print("Three greens:", greens)


# Display palette

In [ ]:
def display_palette(palette: dict) -> None:
    # Display metadata
    for k, v in palette.items():
        if k != "colors":  # Skip colors as we'll display them separately
            print(f'{k}: {v}')

    # Display color swatches
    for c in palette["colors"]:
        img = Image.new(mode="RGB", size=(200, 30), color=tuple(c))
        display(img)

# Final loop through movies

API mode opens each video at the `path` returned by the API (relative to `DRIVE_ROOT` on the mounted Drive). JSON mode scans `MOVIES_DIR` and writes `palettes.json`.

In [ ]:
_VIDEO_SUFFIXES = {".avi", ".mkv", ".mp4"}


def resolve_movie_path(stored_path: str) -> Path:
    """Turn an API path into a local path on the mounted Drive."""
    path = Path(stored_path.strip())
    if path.is_absolute():
        return path
    return DRIVE_ROOT / path


def cleaned_stem(path: Path) -> str:
    """Filename stem with a trailing year and other bracketed tags removed."""
    text = path.stem
    text = re.sub(r"\s*[\(\[]\d{4}[\)\]]", " ", text)
    text = re.sub(r"\s*[\(\[].*?[\)\]]", " ", text)
    return " ".join(text.split())


def list_video_files(root: Path) -> list[Path]:
    if not root.is_dir():
        raise FileNotFoundError(f"Movies folder not found: {root}")
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in _VIDEO_SUFFIXES
    )


def has_recent_palette(palettes: list | None) -> bool:
    """True when a palette is newer than RECALC_IF_BEFORE. None means never skip."""
    if RECALC_IF_BEFORE is None or not palettes:
        return False
    for palette in palettes:
        date_str = palette.get("calculation_date") if isinstance(palette, dict) else None
        if not date_str:
            continue
        if parser.parse(date_str) > RECALC_IF_BEFORE:
            return True
    return False


def rgb_to_hex(color: tuple) -> str:
    red, green, blue = (int(channel) for channel in color)
    return f"#{red:02x}{green:02x}{blue:02x}"


def palette_payload(palette: dict, movie_id: int | None = None) -> dict:
    payload = {
        "id": palette["id"],
        "colors": [rgb_to_hex(color) for color in palette["colors"]],
        "clusters_nb": palette["clusters_nb"],
        "active": True,
        "calculation_date": palette["calculation_date"],
        "calculation_duration_seconds": palette["calculation_duration_seconds"],
        "is_black_and_white": bool(palette["is_black_and_white"]),
        "frame_skip": palette["frame_skip"],
        "resize_width": palette["resize_width"],
        "resize_height": palette["resize_height"],
        "batch_size": palette["batch_size"],
        "clustering_method": palette["clustering_method"],
        "saturation_factor": str(palette["saturation_factor"]),
        "runtime": palette["runtime"],
    }
    if movie_id is not None:
        payload["movie_id"] = movie_id
    else:
        payload["color_space"] = palette["color_space"]
    return payload


def fetch_movies() -> list[dict]:
    response = requests.get(f"{API_BASE_URL.rstrip('/')}/api/movies", timeout=60)
    response.raise_for_status()
    movies = response.json().get("movies", [])
    return [movie for movie in movies if movie.get("type") == "movie"]


def post_palette(palette: dict, movie_id: int) -> None:
    response = requests.post(
        f"{API_BASE_URL.rstrip('/')}/api/movies/palettes",
        headers={"X-API-Key": API_WRITE_KEY},
        json=palette_payload(palette, movie_id=movie_id),
        timeout=60,
    )
    if response.status_code != 201:
        tqdm.write(f"Failed to save palette ({response.status_code}): {response.text}")


def load_json_store() -> dict:
    if not PALETTES_JSON.is_file():
        return {"movies": []}
    data = json.loads(PALETTES_JSON.read_text())
    data.setdefault("movies", [])
    return data


def save_json_store(store: dict) -> None:
    store["movies"] = sorted(store["movies"], key=lambda item: item["file"])
    PALETTES_JSON.write_text(json.dumps(store, indent=2) + "\n")


def run_api() -> None:
    movies = fetch_movies()
    print(f"Found {len(movies)} movies.")
    no_path = 0
    missing_file = 0
    for movie in (pbar := tqdm(movies)):
        title = movie["title"]
        if MOVIE_TO_PROCESS and title != MOVIE_TO_PROCESS:
            continue
        if has_recent_palette(movie.get("palettes")):
            continue
        stored_path = movie.get("path")
        if not stored_path:
            no_path += 1
            continue
        file_path = resolve_movie_path(stored_path)
        if not file_path.is_file():
            missing_file += 1
            tqdm.write(f'Missing file for "{title}": {file_path}')
            continue
        palette = process_movie(
            movie_title=title, file_path=file_path, progress_bar=pbar
        )
        if palette is None:
            continue
        tqdm.write(f'Calculated palette for "{title}"')
        display_palette(palette)
        post_palette(palette, movie["id"])
    if no_path:
        print(f"Skipped {no_path} movies with no path in the API.")
    if missing_file:
        print(f"Skipped {missing_file} movies whose file was not found on Drive.")


def run_json(files: list[Path]) -> None:
    store = load_json_store()
    by_file = {item["file"]: item for item in store["movies"]}
    print(f"Found {len(files)} video files.")
    for file_path in (pbar := tqdm(files)):
        title = cleaned_stem(file_path)
        if MOVIE_TO_PROCESS and title != MOVIE_TO_PROCESS:
            continue
        relative_file = file_path.relative_to(MOVIES_DIR).as_posix()
        record = by_file.get(relative_file) or {
            "title": title,
            "file": relative_file,
            "palettes": [],
        }
        if has_recent_palette(record.get("palettes")):
            continue
        palette = process_movie(
            movie_title=title, file_path=file_path, progress_bar=pbar
        )
        if palette is None:
            continue
        tqdm.write(f'Calculated palette for "{title}"')
        display_palette(palette)
        record["title"] = title
        record.setdefault("palettes", []).append(palette_payload(palette))
        by_file[relative_file] = record
        store["movies"] = list(by_file.values())
        save_json_store(store)


if BACKEND == "api":
    run_api()
elif BACKEND == "json":
    run_json(list_video_files(MOVIES_DIR))
else:
    raise ValueError('BACKEND must be "api" or "json".')
